In [ ]:
import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import anndata2ri
import rpy2.rinterface_lib.callbacks
import logging
from matplotlib import pylab
import os
import sys
anndata2ri.activate()
import yaml
%reload_ext rpy2.ipython

In [ ]:
pylab.rcParams['figure.figsize'] = (9, 9)
homeDir = os.getenv("HOME")
sys.path.insert(1, homeDir+"/utils/")
from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *


In [ ]:
with open(homeDir+"/utils/config.yaml", 'r') as f:
    analysis_params = yaml.safe_load(f)["analysisParams"]
print(analysis_params)
DS = "sc_Cocultures"

# Load data

In [ ]:
import os
import scanpy as sc

base_path = "/storage_kobe/Projects/spatialTX/singleCell"

# find sample folders that contain a filtered_feature_bc_matrix subdir
sample_dirs = []
for d in os.listdir(base_path):
    sample_dir = os.path.join(base_path, d)
    ffm = os.path.join(sample_dir, "filtered_feature_bc_matrix")
    if os.path.isdir(sample_dir) and os.path.isdir(ffm):
        sample_dirs.append((d, ffm))  # (sample_name, path_to_filtered_feature_bc_matrix)

adatas = {}
for sample_name, ffm_path in sorted(sample_dirs):
    print(f"Reading: {sample_name}  ->  {ffm_path}")

    adata = sc.read_10x_mtx(ffm_path, var_names="gene_symbols", cache=False)
    adata.var_names_make_unique()

    adata.obs["orig.ident"] = sample_name
    adata.obs_names = [f"{bc}_{sample_name}" for bc in adata.obs_names]

    adatas[sample_name] = adata

print(f"\n✅ Loaded {len(adatas)} samples:")
print(list(adatas.keys()))


In [ ]:
sample_name =  "2Weeks_Melanoma"
Weeks2 = sc.read_10x_h5(base_path+"/2Weeks_Melanoma_Verrillo_17Nov25/filtered_feature_bc_matrix.h5")
Weeks2.obs["orig.ident"] = "2Weeks_Melanoma"
Weeks2.obs_names = [f"{bc}_{sample_name}" for bc in Weeks2.obs_names]
Weeks2.var_names_make_unique()

In [ ]:
adata = ad.concat(list(adatas.values())+[Weeks2])
adata

In [ ]:
mappingDIct = {"1_MONTH_MELANOMA_1DIC25":"Melanoma_1Month_CoColture",
               "Pretreatment_Melanoma_Verrillo":"Melanoma",
               "2Weeks_Melanoma":"Melanoma_2Weeks_CoColture",
               "SC_positive_Melanoma_96hours":"Melanoma_96Hrs_CoColture"}

In [ ]:
adata.obs["condition"] = adata.obs["orig.ident"].replace(mappingDIct)

# Basic filters

In [ ]:
min_genes = 100

print(adata.shape)
adata = adata[(adata.X > 0 ).sum(axis = 1) > min_genes].copy()
print(adata.shape)

# 1) Remove doublets


In [ ]:
import anndata2ri
import rpy2.rinterface_lib.callbacks
import logging
anndata2ri.activate()
%load_ext rpy2.ipython
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

In [ ]:
%%R
library(scDblFinder)
set.seed(1)
library(BiocParallel)

In [ ]:
adata.layers["counts"] = adata.X.copy()
#del adata.uns
scDblFinder = rpy2.robjects.packages.importr('scDblFinder')
as_data_frame = rpy2.robjects.r['as.data.frame']

In [ ]:
import hashlib, os
import pandas as pd
import numpy as np

# Create a signature hash from the tissue/sample composition
Xtab = pd.crosstab(adata.obs['orig.ident'], adata.obs['condition'])
SingletsHash = hashlib.sha256(Xtab.sort_index(axis=1).to_csv(index=True).encode('utf-8')).hexdigest()
hashFile = f"./out/cache/Singlets.{SingletsHash}.{DS}.xlsx"
os.makedirs(os.path.dirname(hashFile), exist_ok=True)
if os.path.isfile(hashFile):
    print(f" Precomputed singlets found. Using cached result: {SingletsHash}")
    Singlets = pd.read_excel(hashFile, header=None)[0].tolist()
    classification_df = pd.DataFrame(index=Singlets)
    classification_df["scDblFinder.class"] = "singlet"
else:
    print(f" Running doublet detection and caching result as: {SingletsHash}")
    Singlets = []
    all_classifications = []

    for i in adata.obs['orig.ident'].unique():
        adataLocal = adata[adata.obs['orig.ident'] == i].copy()
        adataLocal.X = adataLocal.X.astype(np.float32)
        adataLocal.layers['counts'] = adataLocal.X.copy()

        # Purge, run detection, and convert to R
        adataLocal = PurgeAdata(adataLocal)
        del adataLocal.obs, adataLocal.var

        sce_local = anndata2ri.py2rpy(adataLocal)
        sce_local = scDblFinder.scDblFinder(sce_local)

        # Extract classification
        df_local = pd.DataFrame({
            "scDblFinder.class": sce_local.obs["scDblFinder.class"]
        }, index=sce_local.obs_names)

        all_classifications.append(df_local)

        singlet_ids = df_local.index[df_local["scDblFinder.class"] == "singlet"].tolist()
        Singlets.extend(singlet_ids)
        print(f" Finished doublet detection for: {i}")

    # Save singlets only
    pd.Series(Singlets).to_excel(hashFile, index=False, header=False)

    # Combine classification for attaching later
    classification_df = pd.concat(all_classifications)
    classification_df = classification_df.loc[adata.obs_names]  # align

# Attach classification to adata.obs
print(f"Pre-filter Adata: {adata.shape}")
print(f"Number of Singlets: {(classification_df['scDblFinder.class'] == 'singlet').sum()}")
adata.obs["scDblFinder.class"] = classification_df["scDblFinder.class"]

# Filter to singlets
adataDoublets = adata[adata.obs["scDblFinder.class"] == "doublet"].copy()
adata = adata[adata.obs["scDblFinder.class"] == "singlet"].copy()
print(f"Post-filter Adata: {adata.shape}")

# 2) Other QCs

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

### MT and RP

In [ ]:
FilterGroup = "orig.ident"
import math
goodBClist = {}

########################################################### RIBO genes

fig, axes = plt.subplots(math.ceil(len(adata.obs[FilterGroup].unique())/4),4, figsize=(25, 1*len(adata.obs[FilterGroup].unique())), sharey=False)
plt.subplots_adjust(hspace=1)
excess_axes_count = len(axes.flatten()) - len(adata.obs[FilterGroup].unique())



var = "pct_counts_ribo"
n = 0
goodBClist[var] = []
for i in adata.obs[FilterGroup].unique():
    AdataTMP = adata[adata.obs[FilterGroup] == i].copy()
    mean =  AdataTMP.obs[var].mean()
    mad = abs((AdataTMP.obs[var] - mean)).sum() / AdataTMP.shape[0]
    max = mean+2*mad
    axes.flatten()[n] = sc.pl.violin(AdataTMP, keys=var, show=False, ax = axes.flatten()[n])
    axes.flatten()[n].axhline(y=max, xmin=0, xmax=1, color="red")
    axes.flatten()[n].title.set_text("Sample {} {}".format(i, var))
    goodBClist[var].extend(AdataTMP.obs_names[AdataTMP.obs[var] <= max].tolist())
    n = n+1
goodBClist[var] = set(goodBClist[var])

for i in range(excess_axes_count):
    fig.delaxes(axes.flatten()[-(i+1)])


########################################################### MITO genes
fig, axes = plt.subplots(math.ceil(len(adata.obs[FilterGroup].unique())/4),4, figsize=(25, 1*len(adata.obs[FilterGroup].unique())), sharey=False)
plt.subplots_adjust(hspace=1)
var = "pct_counts_mt"
goodBClist[var] = []
n = 0
#goodBClist = []
for i in adata.obs[FilterGroup].unique():
    AdataTMP = adata[adata.obs[FilterGroup] == i].copy()
    mean =  AdataTMP.obs[var].mean()
    mad = abs((AdataTMP.obs[var] - mean)).sum() / AdataTMP.shape[0]
    #max = mean+10*mad
    max = mean+2*mad
    axes.flatten()[n] = sc.pl.violin(AdataTMP, keys=var, show=False, ax = axes.flatten()[n])
    axes.flatten()[n].axhline(y=max, xmin=0, xmax=1, color="red")
    axes.flatten()[n].title.set_text("Sample {} {}".format(i, var))
    goodBClist[var].extend(AdataTMP.obs_names[AdataTMP.obs[var] <= max].tolist())
    n = n+1
goodBClist[var] = set(goodBClist[var])


for i in range(excess_axes_count):
    fig.delaxes(axes.flatten()[-(i+1)])



In [ ]:

adataPrefilt = adata[list(set.intersection(*list(goodBClist.values())))]
NumberGoodCells = adataPrefilt.shape[0]
GoodCellsRate = adataPrefilt.shape[0]/adata.shape[0]
print("Thresholding like this will results in {} cells kept".format(NumberGoodCells))
print("Which corresponde to {} cells rate".format(round(GoodCellsRate,2)))

In [ ]:
# Axctual filtering
adata = adata[adataPrefilt.obs_names]
del adataPrefilt

### Counts

In [ ]:
import math
fig, axes = plt.subplots(math.ceil(len(adata.obs[FilterGroup].unique())/4),4, figsize=(25, 1*len(adata.obs[FilterGroup].unique())), sharey=False)
adata.obs["log_total_counts"] = np.log(adata.obs["total_counts"])
var = "log_total_counts"
excess_axes_count = len(axes.flatten()) - len(adata.obs[FilterGroup].unique())
plt.subplots_adjust(hspace=1)

n = 0
goodBClist = []
for i in adata.obs[FilterGroup].unique():
    AdataTMP = adata[adata.obs[FilterGroup] == i].copy()
    mean =  AdataTMP.obs[var].mean()
    mad = abs((AdataTMP.obs[var] - mean)).sum() / AdataTMP.shape[0]
    min = mean-2.5*mad
    axes.flatten()[n] = sc.pl.violin(AdataTMP, keys=var, show=False, ax = axes.flatten()[n])
    axes.flatten()[n].axhline(y=mean, xmin=0, xmax=1, color="blue")
    axes.flatten()[n].axhline(y=min, xmin=0, xmax=1, color="green")
    axes.flatten()[n].title.set_text("Sample {}".format(i))
    goodBClist.extend(adata.obs_names[(adata.obs[var] >= min) &  (adata.obs[FilterGroup] == i) ].tolist())
    n = n+1
    
for i in range(excess_axes_count):
    fig.delaxes(axes.flatten()[-(i+1)])



In [ ]:

NumberGoodCells = len(goodBClist)
GoodCellsRate = NumberGoodCells/adata.shape[0]
print("Thresholding like this will results in {} cells kept".format(NumberGoodCells))
print("Which corresponde to {} cells rate".format(round(GoodCellsRate,2)))

In [ ]:
adata.obs["study"] = DS

In [ ]:
# Axctual filtering
adata = adata[goodBClist].copy()

In [ ]:
from scipy.sparse import  csr_matrix, issparse

if issparse(adata.X) == False:
    adata.X = csr_matrix(adata.X)
    print('Converted adata.X to', type(adata.X))

if issparse(adata.layers["counts"]) == False:
    adata.layers["counts"] = csr_matrix(adata.layers["counts"])
    print('Converted counts layer to', type(adata.layers["counts"]))

adata.X = adata.X.astype(np.float32)
adata.layers["counts"] = adata.layers["counts"].astype(np.float32)

In [ ]:
adata.write_h5ad(os.path.join(homeDir, "1_DataPreparation/out/0_Filtered.h5ad"))

In [ ]:
adata.obs["condition"].value_counts()